# Classificação de Produtos com Árvore de Decisão

**Objetivo:** Construir um modelo que, dadas algumas características de um produto (e.g., família, subfamília, histórico de consumo de massa, tempo desde a última manutenção), consiga prever a qual categoria ele pertence (`Cód. Produto`).

## DecisionTreeClassifier

Esse algoritmo cria uma estrutura em árvore onde cada nó representa uma decisão baseada em um atributo do produto. O objetivo é chegar a uma folha da árvore que represente a classe prevista.

**Vantagens:**

* Fácil interpretação
* Lida bem com dados categóricos e numéricos

**Desvantagens:**

* Pode sofrer de overfitting (se ajusta demais aos dados de treino e não generaliza bem para novos dados).

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [4]:
# Load the dataset
df = pd.read_csv("../../Fase02/data/manutencao/dados_manutencao.csv", header=0)

# Rename columns
df = df.rename(
    columns={
        "Qt. Total Acumulada Produzida até a data específica": "Qtd_Produzida",
        "Qt. Acumulada Refugada até a data específica": "Qtd_Refugada",
        "Qtd. Acumulada total Retrabalhada até a data específica": "Qtd_Retrabalhada",
        "Consumo total de Massa Acumulada": "Consumo_Massa_Total",
        "Tempo Restante para Manutenção": "Tempo_Restante_Manutencao",
    }
)

# Assign values to X and y
X = df[
    [
        "Qtd_Produzida",
        "Qtd_Refugada",
        "Qtd_Retrabalhada",
        "Consumo_Massa_Total",
        "Tempo_Restante_Manutencao",
    ]
]
y = df["Cod Produto"]

In [5]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Create transformers
numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])
categorical_transformer = Pipeline(
    steps=[("onehot", OneHotEncoder(handle_unknown="ignore"))]
)

In [6]:
# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            [
                "Qtd_Produzida",
                "Qtd_Refugada",
                "Qtd_Retrabalhada",
                "Consumo_Massa_Total",
                "Tempo_Restante_Manutencao",
            ],
        )
    ]
)

In [7]:
# Create and train the model
model = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", DecisionTreeClassifier())]
)
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Qtd_Produzida',
                                                   'Qtd_Refugada',
                                                   'Qtd_Retrabalhada',
                                                   'Consumo_Massa_Total',
                                                   'Tempo_Restante_Manutencao'])])),
                ('classifier', DecisionTreeClassifier())])

In [8]:
# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Acurácia do modelo: {accuracy:.4f}")

# Print classification report
print(classification_report(y_test, y_pred))

Acurácia do modelo: 0.3100
              precision    recall  f1-score   support

     SA02004       0.19      0.25      0.21        56
     SA02961       0.39      0.42      0.40        65
     SA05780       0.38      0.27      0.31        79

    accuracy                           0.31       200
   macro avg       0.32      0.31      0.31       200
weighted avg       0.33      0.31      0.31       200



## Análise da Acurácia e Melhorias Possíveis

A acurácia do modelo de árvore de decisão é de **31%**, o que significa que ele prevê corretamente o `Cód. Produto` em 31% dos casos no conjunto de teste. Isso não é muito melhor do que um palpite aleatório, então vamos tentar melhorar isso.

###  Estratégias para Melhorar a Acurácia

Aqui estão algumas coisas que podemos tentar:

* **Mais dados:** Geralmente, mais dados levam a melhores resultados. Um conjunto de dados maior e mais diversificado pode ajudar o modelo a aprender padrões mais complexos e generalizar melhor.
* **Recursos diferentes:** Podemos tentar usar colunas diferentes do conjunto de dados como entrada para nosso modelo, ou criar novos recursos a partir dos existentes. A engenharia de recursos pode ser crucial para extrair informações mais relevantes dos dados.
* **Ajuste de hiperparâmetros:** Podemos tentar ajustar os hiperparâmetros do nosso modelo (como `max_depth`, `min_samples_split`, `min_samples_leaf`, etc.) para melhorar seu desempenho. Isso pode ajudar a encontrar a configuração ideal que equilibra o viés e a variância do modelo.
* **Algoritmo diferente:** Podemos tentar usar um algoritmo diferente para construir nosso modelo, como Random Forest, Gradient Boosting, Support Vector Machines (SVM), ou redes neurais. Cada algoritmo tem seus próprios pontos fortes e fracos, e um algoritmo diferente pode ser mais adequado para os seus dados.

Como não há valores ausentes no conjunto de dados, podemos pular a etapa de imputação de valores ausentes e tentar ajustar os hiperparâmetros do nosso modelo.

In [13]:
# Ajustar o hiperparametro da arvore de decisao
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [14]:
# Load the dataset
df = pd.read_csv("../../Fase02/data/manutencao/dados_manutencao.csv", header=0)

# Rename columns
df = df.rename(
    columns={
        "Qt. Total Acumulada Produzida até a data específica": "Qtd_Produzida",
        "Qt. Acumulada Refugada até a data específica": "Qtd_Refugada",
        "Qtd. Acumulada total Retrabalhada até a data específica": "Qtd_Retrabalhada",
        "Consumo total de Massa Acumulada": "Consumo_Massa_Total",
        "Tempo Restante para Manutenção": "Tempo_Restante_Manutencao",
    }
)

# Assign values to X and y
X = df[
    [
        "Qtd_Produzida",
        "Qtd_Refugada",
        "Qtd_Retrabalhada",
        "Consumo_Massa_Total",
        "Tempo_Restante_Manutencao",
    ]
]
y = df["Cod Produto"]

In [15]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define the parameter grid
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [2, 3, 4, 5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

# Create the Decision Tree model
dtc = DecisionTreeClassifier(random_state=42)

# Create the GridSearchCV object
grid_search = GridSearchCV(estimator=dtc, param_grid=param_grid, cv=5)

In [16]:
# Fit the GridSearchCV object to the training data
grid_search.fit(X_train, y_train)

# Get the best parameters
best_params = grid_search.best_params_
print(f"Best parameters: {best_params}")

# Use the best model to make predictions on the test data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Evaluate the best model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy of best model: {accuracy:.4f}")

# Print classification report
print(classification_report(y_test, y_pred))

Best parameters: {'criterion': 'gini', 'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2}
Accuracy of best model: 0.2950
              precision    recall  f1-score   support

     SA02004       0.20      0.29      0.23        56
     SA02961       0.38      0.38      0.38        65
     SA05780       0.34      0.23      0.27        79

    accuracy                           0.29       200
   macro avg       0.31      0.30      0.30       200
weighted avg       0.31      0.29      0.30       200



## Resultados do Ajuste de Hiperparâmetros e Próximos Passos

Os melhores parâmetros encontrados pelo `GridSearchCV` para o modelo de árvore de decisão são:

* **critério:** 'gini'
* **profundidade máxima:** 20
* **min_samples_leaf:** 2
* **min_samples_split:** 2

No entanto, mesmo com esses parâmetros, a precisão do modelo permaneceu em **29,50%**, o que não é uma melhoria significativa em relação ao modelo anterior.

O relatório de classificação mostra que o modelo tem dificuldade em classificar todas as três categorias de produtos (`SA02004`, `SA02961` e `SA05780`), com pontuações de precisão, recall e f1-score relativamente baixas para todas elas.

###  Considerações e Próximos Passos

Isso sugere que uma árvore de decisão pode não ser o modelo mais adequado para este conjunto de dados e tarefa de classificação.  

Algumas alternativas a serem consideradas:

* **Testar outros algoritmos:**  Talvez outros algoritmos, como **Random Forest**, **Gradient Boosting** ou **Support Vector Machines (SVM)**, possam ter um desempenho melhor.
* **Engenharia de recursos:** Vale a pena considerar a engenharia de recursos, ou seja, criar novos recursos a partir dos existentes ou tentar usar diferentes colunas do conjunto de dados como entrada para o modelo.
* **Coletar mais dados:** Como mencionado anteriormente, mais dados geralmente levam a melhores resultados, então coletar mais dados pode ser outra forma de melhorar a precisão do modelo.